# 04 — RAG Avançado com ChromaDB

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 03_RAG  
**Ambiente:** `eai07` (Python 3.11)

---

## O que mudou em relação à versão `.pkl`

| Aspecto | Versão pkl (FAISS) | Esta versão (ChromaDB) |
|---|---|---|
| Persistência | `pickle.dump` → arquivo binário | Banco vetorial em diretório local |
| Filtro por módulo | Recria índice FAISS temporário | `where={"modulo": ...}` nativo |
| Reindexar chunks novos | Reconstrói tudo | `collection.upsert()` incremental |
| Inspecionar chunks | `pickle.load` + loop | API query / get diretamente |
| Embeddings externos | `modelo_emb.encode()` → FAISS | Pode usar embedding function do Chroma ou externos |

## O que você vai aprender

- **ChromaDB persistente** — banco vetorial salvo em disco, recarrega sem reprocessar
- **Metadados nativos** — `modulo`, `titulo`, `arquivo` armazenados com o chunk
- **Filtro por módulo** — `where={"modulo": "EAI_01"}` sem recriar índice
- **Upsert incremental** — adiciona/atualiza chunks sem reconstruir tudo
- **Query Expansion** — o LLM reformula a pergunta para melhorar o recall
- **Reranking** — o LLM reordena resultados por relevância real
- **Histórico de conversa** — contexto mantido entre perguntas

---

> 💡 ChromaDB persiste os embeddings em disco como um banco SQLite + arquivos de índice.
> Na segunda execução, basta abrir a collection existente — zero reprocessamento.

## Instalação (executar uma vez)

```bash
conda activate eai07
pip install chromadb
```

## Setup — imports e clientes

In [1]:
import sys, os, json, time, re
import numpy as np
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

modelo_emb = SentenceTransformer('all-MiniLM-L6-v2')

llm = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)
LLM_MODEL    = os.getenv('LLM_MODEL', 'deepseek-chat')
PROJETO_BASE = os.path.abspath('../..')

print(f'LLM    : {LLM_MODEL}')
print(f'Projeto: {PROJETO_BASE}')
print(f'ChromaDB: {chromadb.__version__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM    : deepseek-chat
Projeto: C:\Users\pcwin\Documents\Especialista_em_AI
ChromaDB: 0.4.13


## Funções de processamento de chunks

Idênticas ao notebook anterior — copiadas para este ser executável de forma independente.

In [2]:
ENRIQUECIMENTO = {
    'regressão linear'       : 'regressão linear, ajuste de curva, reta, mínimos quadrados, ajustar linha, coeficientes a e b',
    'mínimos quadrados'      : 'mínimos quadrados, regressão linear, ajuste de reta, least squares',
    'MSE'                    : 'MSE, Mean Squared Error, erro quadrático médio',
    'CNN'                    : 'CNN, rede convolucional, convolutional neural network, redes convolucionais, conv2D',
    'LSTM'                   : 'LSTM, Long Short-Term Memory, células de memória, gates, sequências',
    'deep learning'          : 'deep learning, aprendizado profundo, redes neurais profundas, DL',
    'ANN'                    : 'ANN, rede neural artificial, perceptron, MLP',
    'GRU'                    : 'GRU, Gated Recurrent Unit, células recorrentes',
    'KNN'                    : 'KNN, K-Nearest Neighbors, vizinhos mais próximos',
    'Random Forest'          : 'Random Forest, floresta aleatória, ensemble, árvores de decisão',
    'SVM'                    : 'SVM, Support Vector Machine',
    'embeddings'             : 'embeddings, word embeddings, vetores de palavras, representação vetorial',
    'TF-IDF'                 : 'TF-IDF, term frequency, bag of words, BoW',
    'transformers'           : 'transformers, BERT, GPT, attention, mecanismo de atenção',
    'autovalores'            : 'autovalores, autovetores, eigenvalues, PCA',
    'transformações lineares': 'transformações lineares, matrizes, rotação, escala, cisalhamento',
    'OpenCV'                 : 'OpenCV, visão computacional, processamento de imagens',
    'YOLO'                   : 'YOLO, YOLOv5, detecção de objetos, object detection',
    'reconhecimento facial'  : 'reconhecimento facial, face recognition, detecção de rostos',
    'RAG'                    : 'RAG, Retrieval Augmented Generation, recuperação de documentos, busca semântica',
    'function calling'       : 'function calling, tool calling, ferramentas, tools, agentes, modelo decide',
    'prompt engineering'     : 'prompt engineering, zero-shot, few-shot, chain-of-thought, CoT',
    'MLflow'                 : 'MLflow, rastreamento de experimentos, experiment tracking',
    'drift'                  : 'drift, data drift, monitoramento, degradação do modelo',
}

def enriquecer_chunk(texto, modulo='', arquivo=''):
    prefixo = f'[{modulo}' + (f' / {arquivo}' if arquivo else '') + '] ' if modulo else ''
    sinonimos = [v for k, v in ENRIQUECIMENTO.items() if k.lower() in texto.lower()]
    resultado = prefixo + texto
    if sinonimos:
        resultado += ' | ' + '; '.join(sinonimos)
    return resultado

def chunk_por_secao(texto):
    chunks, titulo, linhas = [], 'Introdução', []
    for linha in texto.split('\n'):
        if linha.startswith('#'):
            if linhas:
                c = ' '.join(linhas).strip()
                if c: chunks.append({'titulo': titulo, 'conteudo': c})
            titulo, linhas = linha.lstrip('#').strip(), []
        elif linha.strip():
            linhas.append(linha.strip())
    if linhas:
        c = ' '.join(linhas).strip()
        if c: chunks.append({'titulo': titulo, 'conteudo': c})
    return chunks

def processar_agent_context(conteudo, modulo, arquivo=''):
    # Extrai prefixo curto: 'EAI_01_Fundamentos_...' → 'EAI_01'
    import re as _re
    m = _re.match(r'(EAI_\d+)', modulo)
    modulo_prefixo = m.group(1) if m else modulo
    chunks = []
    for s in chunk_por_secao(conteudo):
        resumo = ' '.join(s['conteudo'].split()[:40])
        chunks.append({
            'chunk_busca'    : enriquecer_chunk(f"{s['titulo']}: {resumo}", modulo=modulo, arquivo=arquivo),
            'chunk_contexto' : f"[{modulo} — {s['titulo']}]\n{s['conteudo']}",
            'titulo'         : s['titulo'],
            'modulo'         : modulo,
            'modulo_prefixo' : modulo_prefixo,   # 'EAI_01', 'EAI_07', etc.
            'arquivo'        : arquivo,
        })
    return chunks

def encontrar_agent_contexts(pasta_raiz):
    encontrados, ignorar = [], {'.git', 'venv', '.venv', '__pycache__', 'node_modules'}
    for raiz, dirs, arquivos in os.walk(pasta_raiz):
        dirs[:] = [d for d in dirs if d not in ignorar and not d.startswith('.')]
        if 'AGENT_CONTEXT.md' in arquivos:
            partes = raiz.replace('\\', '/').split('/')
            modulo = next((p for p in partes if p.startswith('EAI_')), os.path.basename(raiz))
            caminho_arquivo = os.path.join(raiz, 'AGENT_CONTEXT.md')
            # Caminho relativo ao PROJETO_BASE para armazenar como metadado
            caminho_rel = os.path.relpath(caminho_arquivo, pasta_raiz).replace('\\', '/')
            encontrados.append((modulo, caminho_arquivo, caminho_rel))
    return sorted(encontrados)

print('Funções de processamento carregadas.')

Funções de processamento carregadas.


## 1. ChromaDB — banco vetorial persistente

O `PersistentClient` salva tudo em disco automaticamente.
Na segunda execução, a collection já existe — apenas abre, sem reindexar.

**Estrutura de diretórios gerada:**
```
EAI_07_AI_Generative/
└── data/
    └── chroma_db/          ← banco ChromaDB (substitui indice_rag.pkl)
        ├── chroma.sqlite3
        └── <uuid>/         ← índice HNSW binário
```

In [3]:
# ── Configuração do banco ──────────────────────────────────────────────────
CHROMA_PATH      = '../data/chroma_db'        # diretório persistente
COLLECTION_NAME  = 'agent_contexts'           # nome da collection

os.makedirs(CHROMA_PATH, exist_ok=True)

# Cliente persistente — cria ou abre banco existente
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# get_or_create: reaproveita collection existente (não reconstrói)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'}  # distância cosine → equivale ao IndexFlatIP com normalize
)

total_existente = collection.count()
print(f'Collection "{COLLECTION_NAME}" aberta.')
print(f'Chunks já indexados: {total_existente}')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Collection "agent_contexts" aberta.
Chunks já indexados: 1763


In [4]:
# ── Indexação (pula automaticamente se o banco já está populado) ───────────

def indexar_projeto(pasta_raiz: str, collection, force: bool = False):
    """
    Varre PROJETO_BASE em busca de AGENT_CONTEXT.md e faz upsert no ChromaDB.

    Usa upsert (não insert): rodar novamente atualiza chunks alterados
    sem duplicar os que não mudaram.

    force=True: força reindexação mesmo se já existem chunks.
    """
    if collection.count() > 0 and not force:
        print(f'Banco já populado ({collection.count()} chunks). Pulando indexação.')
        print('Use indexar_projeto(..., force=True) para forçar reindexação.')
        return

    arquivos = encontrar_agent_contexts(pasta_raiz)
    print(f'Encontrados {len(arquivos)} AGENT_CONTEXT.md em {pasta_raiz}')

    total_chunks = 0
    t0 = time.time()

    for modulo, caminho_abs, caminho_rel in arquivos:
        with open(caminho_abs, 'r', encoding='utf-8') as f:
            conteudo = f.read()

        chunks = processar_agent_context(conteudo, modulo, arquivo=caminho_rel)

        if not chunks:
            continue

        # Gera embeddings para os textos de busca
        textos_busca = [c['chunk_busca'] for c in chunks]
        embs = modelo_emb.encode(
            textos_busca,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=64
        ).tolist()

        # IDs únicos: arquivo relativo (slug) + índice do chunk
        # Usar só o módulo causava colisão quando o mesmo módulo tem vários AGENT_CONTEXT.md
        arquivo_slug = caminho_rel.replace('/', '__').replace('\\', '__').replace('.', '_')
        ids = [f"{arquivo_slug}__chunk_{i:04d}" for i in range(len(chunks))]

        # Metadados armazenados com cada chunk (usados nos filtros)
        metadatas = [
            {
                'modulo'         : c['modulo'],
                'modulo_prefixo' : c['modulo_prefixo'],   # 'EAI_01' — usado nos filtros
                'titulo'         : c['titulo'],
                'arquivo'        : c['arquivo'],
            }
            for c in chunks
        ]

        # documents = texto de contexto completo (o que o LLM vai ler)
        documents = [c['chunk_contexto'] for c in chunks]

        # upsert: insere novos, atualiza existentes pelo ID
        collection.upsert(
            ids=ids,
            embeddings=embs,
            documents=documents,
            metadatas=metadatas,
        )

        total_chunks += len(chunks)
        print(f'  [{modulo}] {len(chunks)} chunks indexados')

    print(f'\nIndexação concluída: {total_chunks} chunks em {time.time()-t0:.1f}s')
    print(f'Total no banco: {collection.count()} chunks')


indexar_projeto(PROJETO_BASE, collection)

Banco já populado (1763 chunks). Pulando indexação.
Use indexar_projeto(..., force=True) para forçar reindexação.


## 2. Busca com filtro por módulo

Com ChromaDB, o filtro é feito via `where={}` — sem recriar índice temporário.

In [5]:
def buscar(
    query: str,
    top_k: int = 5,
    score_minimo: float = 0.3,
    filtro_modulo: str = None
) -> list:
    """
    Busca semântica no ChromaDB com filtro opcional por módulo.

    filtro_modulo: prefixo do módulo, ex: 'EAI_01', 'EAI_03'
    ChromaDB usa distância cosine → score = 1 - distancia (quanto maior, mais relevante).
    """
    emb_q = modelo_emb.encode([query], normalize_embeddings=True).tolist()

    # Monta kwargs opcionais — where só entra se filtro_modulo for informado
    kwargs = dict(
        query_embeddings=emb_q,
        n_results=top_k,
        include=['documents', 'metadatas', 'distances'],
    )

    if filtro_modulo:
        # Filtra pelo prefixo curto armazenado em metadado (ex: 'EAI_01')
        # ChromaDB suporta apenas $eq, $ne, $gt, $gte, $lt, $lte, $in, $nin em metadados
        # $eq	igual a (equal)
        # $ne	diferente de (not equal)
        # $gt	maior que (greater than)
        # $gte	maior ou igual a (greater than or equal)
        # $lt	menor que (less than)
        # $lte	menor ou igual a (less than or equal)
        # $in	está contido em uma lista (ex: {cor: {$in: ["azul", "verde"]}})
        # $nin	não está contido em uma lista (not in)
        
        kwargs['where'] = {'modulo_prefixo': {'$eq': filtro_modulo}}

    resultados_raw = collection.query(**kwargs)

    # Converte para o formato padrão do pipeline
    resultados = []
    for doc, meta, dist in zip(
        resultados_raw['documents'][0],
        resultados_raw['metadatas'][0],
        resultados_raw['distances'][0]
    ):
        # ChromaDB retorna distância cosine (0=idêntico, 2=oposto)
        # Converte para score de similaridade [0, 1]
        score = 1.0 - (dist / 2.0)
        if score >= score_minimo:
            resultados.append({
                'contexto': doc,
                'score'   : score,
                'meta'    : meta,
            })

    return resultados

# ── Teste: com e sem filtro de módulo ─────────────────────────────────────
query = 'como foi implementada a regressão linear?'

print('── Sem filtro ───────────────────────────────────────────')
for r in buscar(query, top_k=3):
    print(f"  [{r['score']:.3f}] {r['meta']}")

print()
print('── Com filtro EAI_01 ────────────────────────────────────')
for r in buscar(query, top_k=3, filtro_modulo='EAI_01'):
    print(f"  [{r['score']:.3f}] {r['meta']}")

── Sem filtro ───────────────────────────────────────────


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [0.766] {'arquivo': 'EAI_07_AI_Generative/03_RAG/AGENT_CONTEXT.md', 'modulo': 'EAI_07_AI_Generative', 'modulo_prefixo': 'EAI_07', 'titulo': 'Query Expansion'}
  [0.764] {'arquivo': 'EAI_02_Machine_Learning/AGENT_CONTEXT.md', 'modulo': 'EAI_02_Machine_Learning', 'modulo_prefixo': 'EAI_02', 'titulo': 'Classificação'}
  [0.763] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': '5. regressao_manual.ipynb'}

── Com filtro EAI_01 ────────────────────────────────────
  [0.763] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': '5. regressao_manual.ipynb'}
  [0.741] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': 'Regressão linear manual'}
  [0.738] {'arquivo': 'EAI_01_

## 3. Query Expansion — o LLM melhora a busca

Antes de buscar, o LLM reformula a pergunta em termos técnicos mais precisos.  
Resolve casos como `"ajustar uma linha"` que não encontrava `"regressão linear"`.

In [6]:
def expandir_query(pergunta: str, historico: list = None) -> str:
    """
    Usa o LLM para reformular a pergunta em termos técnicos mais precisos.

    historico: últimas trocas da conversa.
    O LLM decide SE o histórico é relevante para a pergunta atual:
    - Pergunta de follow-up ('desse projeto', 'nele', 'qual foi a acurácia')
      → usa o histórico para resolver a referência
    - Pergunta nova e independente ('como funciona YOLO no EAI_06?')
      → ignora o histórico, expande apenas com termos técnicos
    """
    if historico:
        ultimas = historico[-4:]  # máximo 2 trocas (user+assistant x2)
        ctx_linhas = []
        for msg in ultimas:
            role = 'Usuario' if msg['role'] == 'user' else 'Assistente'
            ctx_linhas.append(f"{role}: {msg['content'][:300]}")
        contexto_str = '\n\nHistórico recente (use APENAS se a pergunta atual for uma continuação):\n' + '\n'.join(ctx_linhas) + '\n'
    else:
        contexto_str = ''

    prompt = (
        'Você é um especialista em IA. Reformule a pergunta abaixo em termos técnicos '
        'mais precisos para melhorar uma busca semântica em documentação técnica de IA.\n\n'
        'REGRAS:\n'
        '1. Se a pergunta for INDEPENDENTE (menciona explicitamente um módulo, tecnologia ou '
        'tema novo), ignore o histórico e expanda apenas com sinônimos e termos técnicos relacionados.\n'
        '2. Se a pergunta for CONTINUAÇÃO (usa pronomes ou referências como "desse projeto", '
        '"nele", "qual foi a acurácia", sem nomear o assunto explicitamente), use o histórico '
        'para resolver a referência e inclua os termos concretos na query.\n'
        '3. Responda APENAS com a query reformulada, sem explicações. Máximo de 2 linhas.'
        + contexto_str
        + f'\n\nPergunta original: {pergunta}\nQuery reformulada:'
    )

    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=120
    )
    return response.choices[0].message.content.strip()


# ── Testa casos que falhavam com query literal ─────────────────────────────
perguntas_problematicas = [
    'como ajustar uma linha aos pontos de dados?',
    'como o modelo decide qual ferramenta chamar?',
    'como funciona o mecanismo de atenção?',
]

print('Query Expansion:\n')
for p in perguntas_problematicas:
    expandida = expandir_query(p)
    print(f'  Original  : {p}')
    print(f'  Expandida : {expandida}')
    print()

Query Expansion:

  Original  : como ajustar uma linha aos pontos de dados?
  Expandida : técnicas de ajuste de curva e regressão para modelagem de dados, otimização de parâmetros e minimização de erro quadrático médio em conjuntos de pontos experimentais

  Original  : como o modelo decide qual ferramenta chamar?
  Expandida : modelo de IA decisão seleção ferramenta chamada de função mecanismo de escolha

  Original  : como funciona o mecanismo de atenção?
  Expandida : mecanismo de atenção em transformers: funcionamento, cálculo de pesos de atenção, softmax, query-key-value, self-attention e multi-head attention



In [7]:
# Compara busca com query original vs expandida
query_original = 'como ajustar uma linha aos pontos de dados?'
query_expandida = expandir_query(query_original)

print(f'Original : {query_original}')
print(f'Expandida: {query_expandida}\n')

print('── Busca com query ORIGINAL ─────────────────────────────')
for r in buscar(query_original, top_k=2):
    print(f"  [{r['score']:.4f}] {r['meta']}")

print()
print('── Busca com query EXPANDIDA ────────────────────────────')
for r in buscar(query_expandida, top_k=2):
    print(f"  [{r['score']:.4f}] {r['meta']}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Original : como ajustar uma linha aos pontos de dados?
Expandida: técnicas de ajuste de curva e regressão para modelagem de dados, otimização de parâmetros e minimização de erro quadrático médio em conjuntos de pontos experimentais

── Busca com query ORIGINAL ─────────────────────────────
  [0.7649] {'arquivo': 'EAI_07_AI_Generative/06_Projetos_Reais/Assistente_Tecnico_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_07_AI_Generative', 'modulo_prefixo': 'EAI_07', 'titulo': 'FAQ'}
  [0.7383] {'arquivo': 'EAI_04_NLP_Classico/Fundamentos/AGENT_CONTEXT.md', 'modulo': 'EAI_04_NLP_Classico', 'modulo_prefixo': 'EAI_04', 'titulo': '"olá, mundo!" → "olá mundo"'}

── Busca com query EXPANDIDA ────────────────────────────
  [0.7659] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': '5. regressao_manual.ipynb'}
  [0.7337] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 

## 4. Reranking — o LLM reordena os resultados

Depois da busca vetorial, o LLM avalia quais chunks são realmente relevantes.

In [8]:
def rerankar(query: str, resultados: list) -> list:
    """
    Usa o LLM para reordenar os chunks recuperados por relevância real.
    Retorna os mesmos resultados mas reordenados.
    """
    if len(resultados) <= 1:
        return resultados

    chunks_texto = '\n\n'.join(
        f"[{i+1}] {r['meta']}\n{r['contexto'][:300]}..."
        for i, r in enumerate(resultados)
    )

    prompt = f"""\
Pergunta: {query}

Avalie os chunks abaixo por relevância para responder a pergunta.
Responda APENAS com os números em ordem de relevância (ex: 3,1,4,2).
Não inclua explicações.

Chunks:
{chunks_texto}

Ordem por relevância:"""

    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=30
    )
    ordem_str = response.choices[0].message.content.strip()

    try:
        numeros = [int(x) for x in re.findall(r'\d+', ordem_str)]
        numeros = [n for n in numeros if 1 <= n <= len(resultados)]
        todos = list(range(1, len(resultados) + 1))
        faltando = [n for n in todos if n not in numeros]
        ordem_final = numeros + faltando
        return [resultados[i-1] for i in ordem_final]
    except Exception:
        return resultados  # fallback: retorna ordem original


# ── Demonstra reranking ────────────────────────────────────────────────────
query = 'como foi implementada a regressão linear no projeto?'
resultados = buscar(query, top_k=4)

print('── Antes do reranking ───────────────────────────────────')
for i, r in enumerate(resultados):
    print(f"  [{i+1}] score={r['score']:.3f} {r['meta']}")

rerankeados = rerankar(query, resultados)

print()
print('── Depois do reranking ──────────────────────────────────')
for i, r in enumerate(rerankeados):
    print(f"  [{i+1}] score={r['score']:.3f} {r['meta']}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


── Antes do reranking ───────────────────────────────────
  [1] score=0.773 {'arquivo': 'EAI_04_NLP_Classico/AGENT_CONTEXT.md', 'modulo': 'EAI_04_NLP_Classico', 'modulo_prefixo': 'EAI_04', 'titulo': 'Progressão de Complexidade'}
  [2] score=0.765 {'arquivo': 'EAI_02_Machine_Learning/AGENT_CONTEXT.md', 'modulo': 'EAI_02_Machine_Learning', 'modulo_prefixo': 'EAI_02', 'titulo': 'Classificação'}
  [3] score=0.762 {'arquivo': 'EAI_04_NLP_Classico/AGENT_CONTEXT.md', 'modulo': 'EAI_04_NLP_Classico', 'modulo_prefixo': 'EAI_04', 'titulo': 'Estrutura Pedagógica'}
  [4] score=0.760 {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': 'RESUMO EXECUTIVO'}

── Depois do reranking ──────────────────────────────────
  [1] score=0.760 {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': 'RESUMO

## 5. RAG com histórico de conversa

Mantém contexto entre perguntas — o assistente lembra o que foi perguntado antes.

In [9]:
SYSTEM_PROMPT = """\
Você é o Assistente Técnico do projeto ESPECIALISTA_EM_IA de Carlos Henrique.
Módulos: EAI_00 (Install Config), EAI_01 (Fundamentos Matemáticos), EAI_02 (Machine Learning),
EAI_03 (Deep Learning), EAI_04 (NLP Clássico), EAI_05 (NLP Transformers),
EAI_06 (Visão Computacional), EAI_07 (IA Generativa), EAI_08 (MLOps).
Responda em português. Use o contexto fornecido. Seja direto e técnico.
"""

class AssistenteRAG:
    """
    Assistente Técnico com RAG avançado sobre ChromaDB:
    - Query expansion automática
    - Reranking dos resultados
    - Histórico de conversa
    - Filtro opcional por módulo (via metadata do ChromaDB)
    """
    def __init__(self, max_historico: int = 6):
        self.historico     = []
        self.max_historico = max_historico

    def responder(
        self,
        pergunta: str,
        top_k: int = 5,
        filtro_modulo: str = None,
        usar_expansion: bool = True,
        usar_reranking: bool = True,
        verbose: bool = False
    ) -> str:

        # 1. Query expansion com contexto do historico
        # Passa as ultimas trocas para resolver referencias como 'desse projeto', 'nele'
        historico_para_expansao = self.historico[-4:] if self.historico else None
        query_busca = expandir_query(pergunta, historico=historico_para_expansao) if usar_expansion else pergunta
        if verbose:
            print(f'Query busca: {query_busca[:80]}...' if len(query_busca) > 80 else f'Query busca: {query_busca}')

        # 2. Busca semântica no ChromaDB
        resultados = buscar(query_busca, top_k=top_k, filtro_modulo=filtro_modulo)

        # 3. Reranking
        if usar_reranking and len(resultados) > 1:
            resultados = rerankar(pergunta, resultados)

        if verbose:
            print(f'Chunks: {len(resultados)}')
            for r in resultados:
                print(f"  [{r['score']:.3f}] {r['meta']}")

        # 4. Monta contexto
        if resultados:
            contexto = '\n\n---\n\n'.join(r['contexto'] for r in resultados[:3])
            conteudo_usuario = f"CONTEXTO DO PROJETO:\n{contexto}\n\nPERGUNTA: {pergunta}"
        else:
            conteudo_usuario = f"Sem contexto relevante encontrado.\n\nPERGUNTA: {pergunta}"

        # 5. Chama LLM com histórico
        mensagens = [{'role': 'system', 'content': SYSTEM_PROMPT}]
        mensagens.extend(self.historico[-self.max_historico:])
        mensagens.append({'role': 'user', 'content': conteudo_usuario})

        response = llm.chat.completions.create(
            model=LLM_MODEL,
            messages=mensagens,
            temperature=0.2,
            max_tokens=600
        )
        resposta = response.choices[0].message.content

        # 6. Atualiza histórico
        self.historico.append({'role': 'user',      'content': pergunta})
        self.historico.append({'role': 'assistant', 'content': resposta})

        return resposta

    def limpar_historico(self):
        self.historico = []
        print('Histórico limpo.')


print('AssistenteRAG (ChromaDB) definido.')

AssistenteRAG (ChromaDB) definido.


In [10]:
# Testa com perguntas em sequência (histórico)
assistente = AssistenteRAG()

perguntas_em_sequencia = [
    'Qual projeto de deep learning classificou obras de arte?',
    'Qual foi a acurácia desse projeto?',
    'Quais técnicas de aumento de dados foram usadas nele?',
    'Quais foi o algoritimo utilizado no projeto de reconhecimento facial?',
]

for pergunta in perguntas_em_sequencia:
    print(f'\n👤 {pergunta}')
    print('─' * 55)
    resposta = assistente.responder(pergunta, verbose=False)
    print(f'🤖 {resposta}')


👤 Qual projeto de deep learning classificou obras de arte?
───────────────────────────────────────────────────────


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🤖 O projeto que classificou obras de arte foi o **EAI_03_Deep_Learning**.

Ele utilizou **Transfer Learning** com a arquitetura **MobileNetV2** (pré-treinada no ImageNet) e **Fine-tuning** para classificar pinturas do dataset **WikiArt - Painter by Numbers** em **6 a 7 estilos artísticos**, como Expressionismo Abstrato, Cubismo, Barroco e Impressionismo. O modelo atingiu aproximadamente **65% de acurácia**, e foi implantado como uma **aplicação web Flask** para upload de imagens.

👤 Qual foi a acurácia desse projeto?
───────────────────────────────────────────────────────


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🤖 A acurácia do projeto foi de aproximadamente **65%**.

👤 Quais técnicas de aumento de dados foram usadas nele?
───────────────────────────────────────────────────────


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🤖 Com base no contexto fornecido, **não há menção explícita** às técnicas de aumento de dados (data augmentation) utilizadas no projeto.

O contexto apenas lista como sugestão de melhoria futura: *"Aumentar dataset (mais estilos, mais imagens por estilo)"*, mas não descreve quais transformações (rotação, zoom, flip, etc.) foram aplicadas durante o treinamento.

**Resposta direta:** O contexto não informa quais técnicas de aumento de dados foram usadas.

👤 Quais foi o algoritimo utilizado no projeto de reconhecimento facial?
───────────────────────────────────────────────────────


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🤖 Com base no contexto fornecido, o algoritmo utilizado no projeto de reconhecimento facial foi o **FaceNet**, especificamente através da biblioteca **DeepFace**.

**Detalhamento técnico:**
- **Modelo de embeddings**: FaceNet (gera vetores de 128 dimensões)
- **Framework de detecção**: OpenCV DNN (Deep Neural Network)
- **Métrica de similaridade**: Scipy (distância cosseno ou euclidiana)
- **Pipeline**: Captura → Detecção (OpenCV DNN) → Embedding (DeepFace/FaceNet) → Reconhecimento (Scipy)

**Performance reportada**: 97-99% de acurácia, rodando a ~5-10 FPS em CPU.


In [11]:
# Testa filtro por módulo + expansão + reranking
assistente2 = AssistenteRAG()

print('Pergunta específica com filtro EAI_01:\n')
resposta = assistente2.responder(
    'Como calcular a reta que melhor se ajusta a dados de altura e peso?',
    filtro_modulo='EAI_01',
    verbose=True
)
print(f'\n🤖 {resposta}')

Pergunta específica com filtro EAI_01:



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query busca: método dos mínimos quadrados regressão linear simples ajuste de reta dados altur...
Chunks: 5
  [0.809] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': 'Regressão Linear'}
  [0.838] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': '5. regressao_manual.ipynb'}
  [0.808] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': 'Regressão linear manual'}
  [0.816] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': 'Previsão'}
  [0.806] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA'

## 6. Utilitários ChromaDB — inspeção e manutenção

Funcionalidades extras que o ChromaDB oferece além do FAISS+pickle.

In [12]:
# ── Inspecionar o banco ───────────────────────────────────────────────────

def inspecionar_colecao():
    """Resumo do banco: total de chunks por módulo."""
    total = collection.count()
    print(f'Total de chunks: {total}\n')

    # Busca todos os metadados (sem embeddings, sem documentos → rápido)
    todos = collection.get(include=['metadatas'])
    modulos = {}
    for meta in todos['metadatas']:
        m = meta.get('modulo', 'desconhecido')
        modulos[m] = modulos.get(m, 0) + 1

    print('Chunks por módulo:')
    for modulo, count in sorted(modulos.items()):
        print(f'  {modulo:<45} {count:>4} chunks')

inspecionar_colecao()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Total de chunks: 1763

Chunks por módulo:
  EAI_00_Install_Config                           18 chunks
  EAI_01_Fundamentos_Matemática_para_IA           26 chunks
  EAI_02_Machine_Learning                        275 chunks
  EAI_03_Deep_Learning                           344 chunks
  EAI_04_NLP_Classico                            413 chunks
  EAI_05_NLP_com_Transformers                    195 chunks
  EAI_06_Visao_Computacional                     225 chunks
  EAI_07_AI_Generative                           210 chunks
  EAI_08_MLOps_e_Implantação                      57 chunks


In [13]:
# ── Atualizar um módulo específico (upsert incremental) ───────────────────

def atualizar_modulo(pasta_raiz: str, prefixo_modulo: str):
    """
    Reindexação incremental: atualiza apenas os chunks de um módulo específico.
    Útil quando você edita o AGENT_CONTEXT.md de um módulo.

    prefixo_modulo: ex. 'EAI_01', 'EAI_07'
    """
    arquivos = [
        (m, c, r) for m, c, r in encontrar_agent_contexts(pasta_raiz)
        if m.startswith(prefixo_modulo)
    ]

    if not arquivos:
        print(f'Nenhum AGENT_CONTEXT.md encontrado para {prefixo_modulo}')
        return

    for modulo, caminho_abs, caminho_rel in arquivos:
        with open(caminho_abs, 'r', encoding='utf-8') as f:
            conteudo = f.read()

        chunks = processar_agent_context(conteudo, modulo, arquivo=caminho_rel)
        textos_busca = [c['chunk_busca'] for c in chunks]
        embs = modelo_emb.encode(textos_busca, normalize_embeddings=True).tolist()

        arquivo_slug = caminho_rel.replace('/', '__').replace('\\', '__').replace('.', '_')
        ids       = [f"{arquivo_slug}__chunk_{i:04d}" for i in range(len(chunks))]
        metadatas = [{'modulo': c['modulo'], 'modulo_prefixo': c['modulo_prefixo'], 'titulo': c['titulo'], 'arquivo': c['arquivo']} for c in chunks]
        documents = [c['chunk_contexto'] for c in chunks]

        collection.upsert(ids=ids, embeddings=embs, documents=documents, metadatas=metadatas)
        print(f'Módulo {modulo} atualizado: {len(chunks)} chunks')


# Exemplo: atualizar apenas o EAI_07
# atualizar_modulo(PROJETO_BASE, 'EAI_07')

In [14]:
# ── Deletar e recriar o banco (reset completo) ────────────────────────────

def resetar_banco(confirmar: bool = False):
    """
    Apaga toda a collection e reconstrói do zero.
    Use confirmar=True para executar (proteção contra reset acidental).
    """
    global collection
    if not confirmar:
        print('Passe confirmar=True para apagar o banco.')
        return

    chroma_client.delete_collection(COLLECTION_NAME)
    collection = chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={'hnsw:space': 'cosine'}
    )
    print('Banco resetado. Reindexando...')
    indexar_projeto(PROJETO_BASE, collection, force=True)


# resetar_banco(confirmar=True)  # descomente para resetar

---
## Resumo

| Etapa | FAISS + pkl (antes) | ChromaDB (agora) |
|---|---|---|
| Persistência | `pickle.dump` / `pickle.load` | `PersistentClient` automático |
| Filtro por módulo | Recria `IndexFlatIP` temporário | `where={"modulo_prefixo": {"$eq": "EAI_01"}}` |
| Atualizar 1 módulo | Reconstrói tudo | `collection.upsert()` parcial |
| Inspecionar chunks | Loop no pickle | `collection.get(include=['metadatas'])` |
| Espaço de distância | Inner Product (com normalize) | Cosine nativo |
| Score retornado | Similaridade direta | `1 - (distancia / 2)` |

**Pipeline completo**: Query Expansion → Busca ChromaDB (com filtro de módulo) → Reranking → LLM com histórico